# ABR NN Stage 2 — Liberman Colab GPU training

Hyperparameter search + final training for Liberman Stage-2 **MLP** (production path for scenario C).

Full **mlp / cnn / cnn_full** comparison with HP grids runs locally in [`abr_liberman_synapse_comparison.ipynb`](abr_liberman_synapse_comparison.ipynb).

**Prerequisites:** Run the export cell in [`abr_liberman_synapse_comparison.ipynb`](abr_liberman_synapse_comparison.ipynb) locally, then upload `figures/cache/nn_colab_liberman/` to Google Drive (or clone this repo on Colab).

| Split | Role |
|-------|------|
| train | HP tuning (90%) + final fit |
| validate | Early stopping only (official) |
| test | Held-out evaluation (animal × frequency) |

In [ ]:
# @title Setup paths
from pathlib import Path

# Option A: uploaded Drive folder containing manifest.json + parquets
DATA_DIR = Path("/content/drive/MyDrive/nn_colab_liberman")

# Option B: clone repo on Colab and use local export
REPO_DIR = Path("/content/Practicum")
USE_REPO = False  # set True after git clone

if USE_REPO:
    DATA_DIR = REPO_DIR / "figures/cache/nn_colab_liberman"

assert (DATA_DIR / "manifest.json").is_file(), f"Missing pack at {DATA_DIR}"
print("Data dir:", DATA_DIR.resolve())

In [ ]:
# @title Install deps + mount Drive (Colab)
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    !pip -q install torch scikit-learn pyarrow

if USE_REPO and not REPO_DIR.is_dir():
    !git clone https://github.com/YOUR_ORG/Practicum.git {REPO_DIR}

if USE_REPO:
    sys.path.insert(0, str(REPO_DIR))
else:
    sys.path.insert(0, str(DATA_DIR))

import torch
print("CUDA:", torch.cuda.is_available())

In [ ]:
# @title Load exported pack
import json

import pandas as pd

if USE_REPO:
    from utils.nn_colab_train import load_colab_pack, run_all_models, run_mlp_only
else:
    from nn_colab_train import load_colab_pack, run_all_models, run_mlp_only

pack = load_colab_pack(DATA_DIR)
manifest = pack["manifest"]
print(json.dumps(manifest["counts"], indent=2))
print("tabular dim:", manifest["n_tabular"])
display(pd.read_parquet(DATA_DIR / "train.parquet").head())

In [ ]:
# @title [MLP only] HP search + final training + test evaluation (scenario C)
OUT_DIR = DATA_DIR / "results"
mlp_summary = run_mlp_only(DATA_DIR, OUT_DIR, pack=pack, verbose=True)
summary = pd.read_parquet(OUT_DIR / "nn_colab_summary.parquet")
display(summary)

### Optional: all models (prefer local liberman notebook)

Full mlp/cnn/cnn_full HP comparison belongs in [`abr_liberman_synapse_comparison.ipynb`](abr_liberman_synapse_comparison.ipynb). Use this cell only if you cannot run that notebook locally.

In [ ]:
# @title [All models] HP search + final training + test evaluation
OUT_DIR = DATA_DIR / "results"
summary = run_all_models(DATA_DIR, OUT_DIR, verbose=True)
display(summary)

### Re-tune MLP after pack refresh

Re-runs **MLP only** (72-config grid) and merges into `nn_colab_summary.parquet`, leaving any existing CNN rows untouched.

In [ ]:
# @title [MLP only] refresh after strain / feature change
OUT_DIR = DATA_DIR / "results"
mlp_summary = run_mlp_only(DATA_DIR, OUT_DIR, pack=pack, verbose=True)
summary = pd.read_parquet(OUT_DIR / "nn_colab_summary.parquet")
display(summary)

In [ ]:
# @title Download results (Colab)
if IN_COLAB:
    from google.colab import files
    import shutil

    zip_path = "/content/nn_colab_results.zip"
    shutil.make_archive(zip_path.replace(".zip", ""), "zip", OUT_DIR)
    files.download(zip_path)
else:
    print("Results written to", OUT_DIR.resolve())